# Phase 2: CNNs & Spatial Hierarchies (PlantVillage)

Questions 2.4, 2.5, 2.6

In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

device = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: mps


## Dataset: PlantVillage
Split: 70% train, 15% val, 15% test.

In [ ]:
DATA_DIR = '../Datasets/plantvillage_dataset/color'

# Standard transform: resize to 128x128, normalize
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

full_dataset = torchvision.datasets.ImageFolder(root=DATA_DIR, transform=transform)
NUM_CLASSES = len(full_dataset.classes)
print(f'Total images: {len(full_dataset)}')
print(f'Number of classes: {NUM_CLASSES}')
print(f'Classes: {full_dataset.classes}')

N = len(full_dataset)
n_train = int(0.70 * N)
n_val   = int(0.15 * N)
n_test  = N - n_train - n_val

gen = torch.Generator().manual_seed(42)
train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [n_train, n_val, n_test], generator=gen
)
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

Total images: 54305
Number of classes: 38
Classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_

---
## Q2.4 – CNN from Scratch & Receptive Fields


In [ ]:
class PlantCNN(nn.Module):
    """3-block CNN: Conv2d + BatchNorm2d + ReLU + MaxPool2d."""

    def __init__(self, num_classes):
        super().__init__()
        # Block 1
        self.conv1 = nn.Conv2d(3, 8, kernel_size=3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(8)
        self.pool1 = nn.MaxPool2d(2, 2)

        # Block 2
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(16)
        self.pool2 = nn.MaxPool2d(2, 2)

        # Block 3
        self.conv3 = nn.Conv2d(16, 32, kernel_size=3, padding=1, bias=False)
        self.bn3   = nn.BatchNorm2d(32)
        self.pool3 = nn.MaxPool2d(2, 2)

        self.relu = nn.ReLU(inplace=True)
        self.flatten = nn.Flatten()

        # Classifier head
        self.fc1 = nn.Linear(32 * 16 * 16, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        # Block 1
        x = self.pool1(self.relu(self.bn1(self.conv1(x))))
        # Block 2
        x = self.pool2(self.relu(self.bn2(self.conv2(x))))
        # Block 3
        x = self.pool3(self.relu(self.bn3(self.conv3(x))))
        
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
_m = PlantCNN(num_classes=NUM_CLASSES).to(device)
x  = torch.zeros(1, 3, 128, 128, device=device)

print(f"{'Layer':<25} | {'Output Shape':<20}")
print('-' * 48)
print(f"{'Input':<25} | {list(x.shape)}")

with torch.no_grad():
    for name, layer in _m.named_children():
        x = layer(x)
        print(f"{name:<25} | {list(x.shape)}")

del _m  # clean up

Layer                     | Output Shape        
------------------------------------------------
Input                     | [1, 3, 128, 128]
conv1                     | [1, 8, 128, 128]
bn1                       | [1, 8, 128, 128]
pool1                     | [1, 8, 64, 64]
conv2                     | [1, 16, 64, 64]
bn2                       | [1, 16, 64, 64]
pool2                     | [1, 16, 32, 32]
conv3                     | [1, 32, 32, 32]
bn3                       | [1, 32, 32, 32]
pool3                     | [1, 32, 16, 16]
relu                      | [1, 32, 16, 16]
flatten                   | [1, 8192]
fc1                       | [1, 256]
fc2                       | [1, 38]


In [ ]:
# ── Theoretical Receptive Field calculation ─────────────────────────────────
rf = 1
stride = 1

layers = [
    ('Conv1 (k=3, s=1)', 3, 1),
    ('Pool1 (k=2, s=2)', 2, 2),
    ('Conv2 (k=3, s=1)', 3, 1),
    ('Pool2 (k=2, s=2)', 2, 2),
    ('Conv3 (k=3, s=1)', 3, 1),
    ('Pool3 (k=2, s=2)', 2, 2),
]

print(f"{'Layer':<25} {'RF':>6} {'Stride':>8}")
print('-' * 42)
print(f"{'Input':<25} {rf:>6} {stride:>8}")

for name, k, s in layers:
    rf = rf + (k - 1) * stride
    stride = stride * s
    print(f"{name:<25} {rf:>6} {stride:>8}")

print(f"\nTheoretical receptive field of a single activation in the "
      f"final feature map: {rf}×{rf} pixels")

Layer                         RF   Stride
------------------------------------------
Input                          1        1
Conv1 (k=3, s=1)               3        1
Pool1 (k=2, s=2)               4        2
Conv2 (k=3, s=1)               8        2
Pool2 (k=2, s=2)              10        4
Conv3 (k=3, s=1)              18        4
Pool3 (k=2, s=2)              22        8

Theoretical receptive field of a single activation in the final feature map: 22×22 pixels


In [ ]:
model_cls = PlantCNN(num_classes=NUM_CLASSES).to(device)

# ── Train the CNN classifier ────────────────────────────────────────────────
EPOCHS_CLF = 15
LR         = 1e-3

criterion_ce = nn.CrossEntropyLoss()
optimizer_cls = optim.Adam(model_cls.parameters(), lr=LR)

train_losses_cls, val_losses_cls = [], []

for epoch in range(1, EPOCHS_CLF + 1):
    # ── train ──
    model_cls.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer_cls.zero_grad()
        out  = model_cls(imgs)
        loss = criterion_ce(out, labels)
        loss.backward()
        optimizer_cls.step()
        running_loss += loss.item() * imgs.size(0)
    train_loss = running_loss / len(train_loader.dataset)

    # ── val ──
    model_cls.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model_cls(imgs)
            val_loss += criterion_ce(out, labels).item() * imgs.size(0)
    val_loss /= len(val_loader.dataset)

    train_losses_cls.append(train_loss)
    val_losses_cls.append(val_loss)

    if epoch % 5 == 0:
        print(f'Epoch {epoch:3d}/{EPOCHS_CLF} | '
              f'Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f}')

torch.save(model_cls.state_dict(), 'best_model_q2_4.pt')
print('Model saved as best_model_q2_4.pt')

In [ ]:
# ── Load pretrained weights 
MODEL_PATH = 'best_model_q2_4.pt'
if os.path.exists(MODEL_PATH):
    model_cls = PlantCNN(num_classes=NUM_CLASSES).to(device)
    model_cls.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    print(f'Successfully loaded weights from {MODEL_PATH}')
else:
    print(f'Weights file {MODEL_PATH} not found. Please run the training cell above.')

Successfully loaded weights from best_model_q2_4.pt


In [ ]:
# ── Test accuracy  ───────────────────────────────
model_cls.eval()
correct = total = 0
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        preds = model_cls(imgs).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

baseline_acc = correct / total
print(f'Q2.4 Test accuracy (baseline): {baseline_acc*100:.2f}%')

Q2.4 Test accuracy (baseline): 91.94%


---
## Q2.5 – CNN Regression Head

In [12]:
from torch.utils.data import Dataset

class SeverityDataset(Dataset):
    """Wraps a Subset of PlantVillage and assigns a synthetic severity score."""

    def __init__(self, subset, rng_seed=0):
        self.subset = subset
        # Identify 'healthy' class indices
        healthy_classes = {
            i for i, name in enumerate(full_dataset.classes)
            if 'healthy' in name.lower()
        }
        rng = np.random.default_rng(rng_seed)
        labels = [full_dataset.targets[idx] for idx in subset.indices]
        severity = []
        for lbl in labels:
            if lbl in healthy_classes:
                severity.append(0.0)
            else:
                severity.append(float(rng.uniform(0.2, 0.9)))
        self.severity = torch.tensor(severity, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, _ = self.subset[idx]
        return img, self.severity[idx]


train_sev = SeverityDataset(train_dataset, rng_seed=0)
val_sev   = SeverityDataset(val_dataset,   rng_seed=1)
test_sev  = SeverityDataset(test_dataset,  rng_seed=2)

train_loader_sev = DataLoader(train_sev, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader_sev   = DataLoader(val_sev,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader_sev  = DataLoader(test_sev,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Severity dataset sizes: train={len(train_sev)}, val={len(val_sev)}, test={len(test_sev)}')

Severity dataset sizes: train=38013, val=8145, test=8147


In [ ]:
class PlantCNN_Regression(nn.Module):
    """Same CNN backbone as Q2.4, but with a regression head (output=1)."""

    def __init__(self):
        super().__init__()
        # Shared backbone
        self.conv1 = nn.Conv2d(3, 8, kernel_size=3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(8)
        self.pool1 = nn.MaxPool2d(2, 2)

        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(16)
        self.pool2 = nn.MaxPool2d(2, 2)

        self.conv3 = nn.Conv2d(16, 32, kernel_size=3, padding=1, bias=False)
        self.bn3   = nn.BatchNorm2d(32)
        self.pool3 = nn.MaxPool2d(2, 2)

        self.relu = nn.ReLU(inplace=True)
        self.flatten = nn.Flatten()

        # Regression head
        self.head = nn.Sequential(
            nn.Linear(32 * 16 * 16, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid(),   
        )

    def forward(self, x):
        x = self.pool1(self.relu(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu(self.bn2(self.conv2(x))))
        x = self.pool3(self.relu(self.bn3(self.conv3(x))))
        x = self.flatten(x)
        return self.head(x)

In [16]:
model_reg = PlantCNN_Regression().to(device)

In [17]:
EPOCHS_REG = 15
criterion_mse = nn.MSELoss()
optimizer_reg = optim.Adam(model_reg.parameters(), lr=LR)

train_losses_reg, val_losses_reg = [], []

for epoch in range(1, EPOCHS_REG + 1):
    model_reg.train()
    running = 0.0
    for imgs, sev in train_loader_sev:
        imgs, sev = imgs.to(device), sev.to(device)
        optimizer_reg.zero_grad()
        loss = criterion_mse(model_reg(imgs), sev)
        loss.backward()
        optimizer_reg.step()
        running += loss.item() * imgs.size(0)
    train_loss = running / len(train_loader_sev.dataset)

    model_reg.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, sev in val_loader_sev:
            imgs, sev = imgs.to(device), sev.to(device)
            val_loss += criterion_mse(model_reg(imgs), sev).item() * imgs.size(0)
    val_loss /= len(val_loader_sev.dataset)

    train_losses_reg.append(train_loss)
    val_losses_reg.append(val_loss)

    if epoch % 5 == 0:
        print(f'Epoch {epoch:3d}/{EPOCHS_REG} | '
              f'Train MSE: {train_loss:.4f} | Val MSE: {val_loss:.4f}')

Epoch   5/15 | Train MSE: 0.0321 | Val MSE: 0.0332
Epoch  10/15 | Train MSE: 0.0295 | Val MSE: 0.0349
Epoch  15/15 | Train MSE: 0.0261 | Val MSE: 0.0346


In [18]:
# ── Report MAE on the test set ──────────────────────────────────────────────
model_reg.eval()
total_ae = 0.0
n_total  = 0
with torch.no_grad():
    for imgs, sev in test_loader_sev:
        imgs, sev = imgs.to(device), sev.to(device)
        preds = model_reg(imgs)
        total_ae += torch.abs(preds - sev).sum().item()
        n_total  += imgs.size(0)

mae = total_ae / n_total
print(f'Q2.5 Test MAE (Severity Percentage): {mae:.4f}')

Q2.5 Test MAE (Severity Percentage): 0.1371


---
## Q2.6 – Translation Invariance

In [23]:
# ── Load 100 test images (unshifted) ───────────────────────────────────────
test_indices = test_dataset.indices[:100]

# Unshifted transform (same as training)
transform_baseline = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Shifted transform: pad 5px on right & bottom, then crop back to 128x128
# Equivalent to shifting content 5px right and 5px down (padding left/top with 0)
transform_shifted = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
    transforms.Pad((5, 5, 0, 0), fill=0),  # pad left=5, top=5 → shifts content right+down
    transforms.CenterCrop(128),             # crop back to 128x128
])

# Build two datasets with different transforms but the same 100 indices
class TransformSubset(Dataset):
    """Same 100 images with a custom transform."""
    def __init__(self, base_dataset, indices, transform):
        # base_dataset is the full ImageFolder
        self.base    = base_dataset
        self.indices = indices
        self.transform = transform
        # Temporarily override transform
        self._orig_transform = base_dataset.transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        # Bypass the dataset's built-in transform
        from PIL import Image
        orig_idx = self.indices[idx]
        path, label = self.base.samples[orig_idx]
        img = Image.open(path).convert('RGB')
        img = self.transform(img)
        return img, label


ds_baseline = TransformSubset(full_dataset, test_indices, transform_baseline)
ds_shifted  = TransformSubset(full_dataset, test_indices, transform_shifted)

loader_baseline = DataLoader(ds_baseline, batch_size=32, shuffle=False, num_workers=0)
loader_shifted  = DataLoader(ds_shifted,  batch_size=32, shuffle=False, num_workers=0)

model_cls.eval()

def evaluate_accuracy(model, loader):
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return correct / total

acc_baseline = evaluate_accuracy(model_cls, loader_baseline)
acc_shifted  = evaluate_accuracy(model_cls, loader_shifted)

print(f'Q2.6 Accuracy on 100 unshifted test images : {acc_baseline*100:.2f}%')
print(f'Q2.6 Accuracy on 100 shifted  test images  : {acc_shifted*100:.2f}%')
print(f'Performance drop due to 5-pixel shift      : {(acc_baseline - acc_shifted)*100:.2f} pp')

Q2.6 Accuracy on 100 unshifted test images : 94.00%
Q2.6 Accuracy on 100 shifted  test images  : 90.00%
Performance drop due to 5-pixel shift      : 4.00 pp
